# 6.3 Context Managers

**Prerequisites:** 6.1 Exception Handling, 4.3 Function Generators  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What `with` actually does, and why `try/finally` alone is not enough
- The `__enter__` / `__exit__` protocol
- What returning `True` from `__exit__` means
- **`@contextlib.contextmanager`** — the generator form
- `suppress`, `closing`, `nullcontext`, **`ExitStack`**
- Multiple context managers, and the parenthesised form (3.10+)
- Real uses: transactions, locks, timers, temporary state

---

## 1. The problem `with` solves

Any time you acquire something, you must release it — a file handle, a lock, a database
connection, a temporary directory. And you must release it **even if the code in between
raises**.

The manual version:

```python
f = open("data.txt")
try:
    process(f)
finally:
    f.close()          # runs whatever happens
```

That is correct, but it has three problems:

1. **It is four lines of ceremony** for one line of work.
2. **It is easy to forget** — and forgetting is silent until you run out of file handles.
3. **It does not compose.** Three resources means three nested `try/finally` blocks.

A **context manager** packages acquire-and-release into an object, so the caller writes one
line and cannot forget the cleanup.

```python
with open("data.txt") as f:
    process(f)
```

### What `with` actually does

```
with EXPRESSION as NAME:
    BODY
```

is roughly:

```python
mgr = EXPRESSION
NAME = mgr.__enter__()
try:
    BODY
finally:
    mgr.__exit__(exc_type, exc_value, traceback)
```

Two things follow immediately:
- **`as NAME` receives whatever `__enter__` returns** — not the manager itself, unless it
  returns `self`.
- **`__exit__` always runs**, and it is told whether an exception occurred.

In [ ]:
# A context manager is any object with __enter__ and __exit__.

class Timer:
    """Times the block it wraps."""

    def __init__(self, label: str) -> None:
        self.label = label
        self.elapsed = 0.0

    def __enter__(self):
        import time
        print(f"  [{self.label}] start")
        self._start = time.perf_counter()
        return self                      # this is what `as` binds

    def __exit__(self, exc_type, exc_value, tb):
        import time
        self.elapsed = time.perf_counter() - self._start
        status = "ok" if exc_type is None else f"failed with {exc_type.__name__}"
        print(f"  [{self.label}] {status} after {self.elapsed * 1000:.2f} ms")
        return False                     # do NOT suppress exceptions


# Success path
with Timer("sum") as t:
    total = sum(range(200_000))
print("elapsed recorded:", round(t.elapsed, 6), "s\n")


# Failure path - __exit__ still runs, and sees the exception
try:
    with Timer("failing") as t:
        raise ValueError("something broke")
except ValueError as exc:
    print("exception propagated out:", exc)


# ---- What __exit__ receives ----
class Inspect:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, tb):
        print(f"\n  exc_type : {exc_type}")
        print(f"  exc_value: {exc_value!r}")
        print(f"  traceback: {'present' if tb else 'None'}")
        return False

with Inspect():
    pass

try:
    with Inspect():
        1 / 0
except ZeroDivisionError:
    print("  (and the exception continued on its way)")

### ⚠️ Returning `True` from `__exit__` swallows the exception

`__exit__`'s return value is a **flag meaning "I handled it"**:

| Returns | Effect |
|---|---|
| `False` (or `None`) | The exception propagates normally — **this is what you want almost always** |
| `True` | The exception is **suppressed** and execution continues after the `with` |

This is the one genuinely surprising part of the protocol, and it is a common accidental
bug: a `__exit__` that ends with a `print()` returns `None`, which is falsy, which is
correct. But a `__exit__` that ends with `return self.cleanup()` might return something
truthy — and silently swallow every exception in the block.

**Be explicit: end `__exit__` with `return False` unless you genuinely mean to suppress.**

In [ ]:
class Swallow:
    """⚠️ Suppresses everything - almost never what you want."""

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, tb):
        if exc_type is not None:
            print(f"  swallowed {exc_type.__name__}: {exc_value}")
        return True                       # ⚠️ suppresses the exception


with Swallow():
    raise ValueError("this never reaches the caller")

print("execution continues here - the ValueError vanished\n")


# ---- Selective suppression: the legitimate use ----
class IgnoreMissing:
    """Suppresses FileNotFoundError only, and lets everything else through."""

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, tb):
        return exc_type is not None and issubclass(exc_type, FileNotFoundError)


with IgnoreMissing():
    raise FileNotFoundError("optional.cfg")
print("missing optional file ignored")

try:
    with IgnoreMissing():
        raise PermissionError("secrets.cfg")
except PermissionError as exc:
    print("PermissionError still propagates:", exc)


# ---- The accidental version ----
class AccidentallySwallows:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, tb):
        return self.cleanup()             # ⚠️ returns a truthy value

    def cleanup(self):
        return "done"                     # a non-empty string is truthy!

with AccidentallySwallows():
    raise RuntimeError("silently discarded")

print("\n⚠️ RuntimeError was swallowed because cleanup() returned a truthy string.")

---

## 2. `@contextmanager` — the generator form

Writing a class for every context manager is heavy. `contextlib.contextmanager` turns a
**generator** into one: everything before the `yield` is `__enter__`, everything after is
`__exit__`.

### Syntax breakdown

```
@contextmanager
def managed():
    setup()                 <- __enter__
    try:
        yield resource      <- what `as` binds; the body runs here
    finally:
        teardown()          <- __exit__, guaranteed
```

⚠️ **The `try/finally` is not optional.** If the body raises, the exception is thrown *into*
the generator at the `yield` (see `generator.throw()` in **4.3**). Without `finally`, your
teardown never runs — which defeats the whole purpose.

In [ ]:
from contextlib import contextmanager


# ---- A database transaction: commit on success, roll back on failure ----
class FakeConnection:
    def __init__(self) -> None:
        self.log: list[str] = []
        self.committed = False

    def execute(self, sql: str) -> None:
        self.log.append(sql)

    def commit(self) -> None:
        self.committed = True
        self.log.append("COMMIT")

    def rollback(self) -> None:
        self.log.append("ROLLBACK")


@contextmanager
def transaction(conn: FakeConnection):
    """Commit if the block succeeds, roll back if it raises."""
    conn.execute("BEGIN")
    try:
        yield conn                       # the `with` body runs here
    except Exception:
        conn.rollback()
        raise                            # re-raise: we cleaned up, we did not handle it
    else:
        conn.commit()


# Success
conn = FakeConnection()
with transaction(conn) as tx:
    tx.execute("INSERT INTO orders VALUES (1, 250)")
    tx.execute("UPDATE stock SET qty = qty - 1")
print("success :", conn.log)

# Failure
conn = FakeConnection()
try:
    with transaction(conn) as tx:
        tx.execute("INSERT INTO orders VALUES (2, 100)")
        raise RuntimeError("payment gateway timed out")
except RuntimeError as exc:
    print("failure :", conn.log, "|", exc)


# ---- ⚠️ Without try/finally, cleanup is skipped on failure ----
@contextmanager
def broken():
    print("\n  acquire")
    yield
    print("  release")                   # never runs if the body raises

try:
    with broken():
        raise ValueError("boom")
except ValueError:
    print("  <- notice 'release' never printed")


@contextmanager
def correct():
    print("\n  acquire")
    try:
        yield
    finally:
        print("  release")               # always runs

try:
    with correct():
        raise ValueError("boom")
except ValueError:
    print("  <- release ran, as it must")

In [ ]:
from contextlib import contextmanager
import os


# ---- Temporarily changing global state, and always putting it back ----
@contextmanager
def env(**overrides: str):
    """Temporarily set environment variables."""
    original = {k: os.environ.get(k) for k in overrides}
    os.environ.update(overrides)
    try:
        yield
    finally:
        for key, value in original.items():
            if value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = value


print("before :", os.environ.get("APP_MODE"))
with env(APP_MODE="test", APP_DEBUG="1"):
    print("inside :", os.environ.get("APP_MODE"), os.environ.get("APP_DEBUG"))
print("after  :", os.environ.get("APP_MODE"))


# ---- Same shape, for any mutable state ----
@contextmanager
def temporary(container: dict, **overrides):
    saved = {k: container.get(k) for k in overrides}
    container.update(overrides)
    try:
        yield container
    finally:
        for k, v in saved.items():
            if v is None:
                container.pop(k, None)
            else:
                container[k] = v


config = {"retries": 3, "timeout": 30}
print("\nbefore:", config)
with temporary(config, retries=0, timeout=1) as cfg:
    print("inside:", cfg)
print("after :", config)

# ...and it restores even when the body raises
try:
    with temporary(config, retries=99):
        raise RuntimeError("failure inside the block")
except RuntimeError:
    print("after failure:", config)

---

## 3. The `contextlib` toolkit

Five utilities that cover most of what you would otherwise hand-write.

| Tool | Purpose |
|---|---|
| `suppress(*exceptions)` | Ignore specific exceptions — an explicit `try/except/pass` |
| `closing(thing)` | Call `.close()` on something that lacks `__exit__` |
| `nullcontext(value)` | A do-nothing manager — for optional resources |
| `ExitStack()` | Manage a **dynamic** number of context managers |
| `redirect_stdout(f)` | Capture output from code you cannot modify |

**`ExitStack` is the one worth knowing about.** A `with` statement needs to know its
resources at write time. When the number is decided at run time — open every file in a
directory, acquire one lock per shard — `ExitStack` handles it, unwinding in reverse order.

In [ ]:
from contextlib import suppress, closing, nullcontext, ExitStack, redirect_stdout
import io


# ---- suppress: explicit, narrow, self-documenting ----
config = {"timeout": 30}

with suppress(KeyError):
    print("missing key:", config["retries"])
print("continued past the missing key")

# It only suppresses what you named
try:
    with suppress(KeyError):
        1 / 0
except ZeroDivisionError:
    print("ZeroDivisionError not suppressed - correct")


# ---- closing: for objects with .close() but no __exit__ ----
class LegacyCursor:
    def __init__(self) -> None:
        self.open = True

    def close(self) -> None:
        self.open = False
        print("  cursor closed")


cursor = LegacyCursor()
with closing(cursor) as cur:
    print("\n  using cursor, open =", cur.open)
print("  after block, open =", cursor.open)


# ---- nullcontext: an optional resource, without duplicating the body ----
def process(rows, log_file=None):
    # Either a real file, or a manager that does nothing
    manager = open(log_file, "w") if log_file else nullcontext()
    with manager as handle:
        for row in rows:
            if handle:
                handle.write(f"{row}\n")
    return len(rows)

print("\nwithout logging:", process([1, 2, 3]))


# ---- ExitStack: a number of resources decided at run time ----
class Resource:
    def __init__(self, name: str) -> None:
        self.name = name

    def __enter__(self):
        print(f"  open  {self.name}")
        return self

    def __exit__(self, *exc):
        print(f"  close {self.name}")
        return False


names = ["shard-a", "shard-b", "shard-c"]      # length known only at run time

print("\nExitStack:")
with ExitStack() as stack:
    resources = [stack.enter_context(Resource(n)) for n in names]
    print("  working with", [r.name for r in resources])
print("  ^ closed in REVERSE order, and would be even if the body raised")


# ---- redirect_stdout: capture output from code you cannot change ----
def noisy():
    print("this would normally hit the terminal")

buffer = io.StringIO()
with redirect_stdout(buffer):
    noisy()
print("\ncaptured:", buffer.getvalue().strip())

---

## 4. Multiple context managers

Several resources in one `with`, separated by commas — equivalent to nesting, and they
unwind in reverse order:

```python
with open("in.txt") as src, open("out.txt", "w") as dst:
    dst.write(src.read())
```

> **Version note (3.10+):** you can wrap the list in parentheses to split it across lines.
> Before 3.10 this was a `SyntaxError` and people used backslashes or `ExitStack`.
>
> ```python
> with (
>     open("in.txt") as src,
>     open("out.txt", "w") as dst,
> ):
>     ...
> ```

### Two caveats

- **Most context managers are not reusable.** A generator-based one is exhausted after a
  single `with`; entering it a second time **fails**. The exact exception varies by CPython
  version (`RuntimeError` on some, `AttributeError` on 3.14), so never rely on catching a
  particular type — just call the factory again each time.
- **Most are not re-entrant either** — you cannot nest the *same* manager object inside
  itself. `contextlib.reentrant` variants and `threading.RLock` exist for when you need it.

In [ ]:
from contextlib import contextmanager


class Res:
    def __init__(self, name): self.name = name
    def __enter__(self):
        print(f"  enter {self.name}"); return self
    def __exit__(self, *exc):
        print(f"  exit  {self.name}"); return False


# Comma-separated: equivalent to nesting
print("comma form:")
with Res("outer") as a, Res("inner") as b:
    print("   body")

# Parenthesised form (3.10+) - same thing, splittable across lines
print("\nparenthesised form (3.10+):")
with (
    Res("first") as first,
    Res("second") as second,
):
    print("   body")


# ---- ⚠️ Generator-based managers are single-use ----
@contextmanager
def single_use():
    yield "value"

mgr = single_use()

with mgr as value:
    print("\nfirst use :", value)

# The second entry fails. The exception TYPE differs across CPython versions,
# so catch broadly here - the lesson is "it fails", not "it raises X".
try:
    with mgr as value:
        print("second use:", value)
except Exception as exc:
    print(f"second use: {type(exc).__name__} - {exc}")

print("\nFix: call the factory again each time")
with single_use() as v1, single_use() as v2:
    print("  two independent managers:", v1, v2)


# ---- Class-based managers CAN be reusable, if you write them that way ----
class Reusable:
    def __enter__(self):
        print("  enter"); return self
    def __exit__(self, *exc):
        print("  exit"); return False

r = Reusable()
print("\nreusable class:")
with r: pass
with r: pass

---

## Common Mistakes & Pitfalls

1. **Returning a truthy value from `__exit__` by accident.** It silently suppresses every exception in the block. End with an explicit `return False`.
2. **Omitting `try/finally` inside a `@contextmanager` generator.** If the body raises, the code after `yield` never runs and the resource leaks.
3. **`yield`ing more than once in a `@contextmanager` function.** It must yield exactly once; more raises `RuntimeError`.
4. **Reusing a generator-based context manager.** They are single-use — call the factory again.
5. **Assuming `as` binds the manager.** It binds whatever `__enter__` *returns*. `open()` returns the file, but your own class must `return self` if that is what you want.
6. **Using `try/finally` where a context manager belongs.** If you write the same acquire/release pair twice, make it a context manager.
7. **`except Exception: pass` where `suppress` says it better.**
8. **Nesting five `with` statements** when the count is dynamic — that is what `ExitStack` is for.

## Best Practices

- Use `with` for **every** resource that must be released — files, locks, connections, sockets, temp directories.
- Write `@contextmanager` generators for simple cases; a class when the manager needs state or methods.
- Always wrap the `yield` in `try/finally`.
- End `__exit__` with an explicit `return False` unless suppression is the point.
- Use `contextlib.suppress` instead of `try/except/pass`.
- Use `ExitStack` when the number of resources is decided at run time.
- Prefer a context manager over `finally` for cleanup — it is reusable and cannot be forgotten at the call site.
- Name them for what they *do*: `transaction()`, `timed()`, `temporary_env()`.

## Practice Exercises

Try these before moving on.

1. Write a `Timer` context manager that reports elapsed time, and works when the body raises.
2. Write `@contextmanager def chdir(path)` that changes directory and always changes back.
3. Write a context manager that opens a file and deletes it afterwards, even on failure.
4. Show a `__exit__` that accidentally returns a truthy value, and prove it swallows errors.
5. Use `ExitStack` to open every `.txt` file in a directory, however many there are.
6. Write a `retry` context manager... then explain why a **decorator** is the better tool for retries, and a context manager for resources.
7. Use `redirect_stdout` to capture the output of a function you are not allowed to modify.
8. Rewrite a `try/finally` block from **6.1** as a reusable context manager.